In [1]:
import json
import os
import urllib3
from sqlalchemy import create_engine, text

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = os.getenv("MES_BASE_URL", "https://localhost:7204")
SWAGGER_URL = f"{BASE_URL}/swagger/v1/swagger.json"


DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321",
}

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)



In [2]:
import pandas as pd
from sqlalchemy import text

query = """
SELECT *
FROM backend_api_table
WHERE LOWER(access_to_use) = 'allowed';
"""

df = pd.read_sql(query, engine)

print(f"Total Allowed APIs: {len(df)}")
display(df.head(5))

Total Allowed APIs: 362


,api_id,method,end_point,description,access_to_use,parameter,error_status,created_at
0,f064f38b-4e61-4a61-ab0c-adffdfcd18d8,GET,/api/v1/account,Executes GetStatus. Usable by AccountApi agents.,allowed,"{'priority': 100, 'tool_name': 'get_status', '...","{'status': 'discovered', 'message': 'ingested ...",2026-08-07 06:33:04.032830+00:00
1,d1568d06-8dec-4974-9a0d-e7e836137bd7,GET,/api/v1/account/roles,Executes GetRoles. Usable by AccountApi agents.,allowed,"{'priority': 100, 'tool_name': 'get_roles', 'r...","{'status': 'discovered', 'message': 'ingested ...",2026-08-07 06:33:04.032830+00:00
2,c32f16e2-7800-48fe-bbb3-29fba687d4c2,GET,/api/v1/account/menu,Executes GetMenus. Usable by AccountApi agents.,allowed,"{'priority': 100, 'tool_name': 'get_menus', 'r...","{'status': 'discovered', 'message': 'ingested ...",2026-08-07 06:33:04.032830+00:00
3,ff138d2e-3ce6-4999-ab63-124d543a5ef0,GET,/api/v1/account/user-by-id/{id},Executes UserById. Usable by AccountApi agents.,allowed,"{'priority': 100, 'tool_name': 'user_by_id', '...","{'status': 'discovered', 'message': 'ingested ...",2026-08-07 06:33:04.032830+00:00
4,f4f4917d-d61c-4cee-9039-c3fb36c17e0b,GET,/api/v1/production/alerts/get-alert,Executes GetAlertScreen. Usable by ProductionD...,allowed,"{'priority': 100, 'tool_name': 'get_alert_scre...","{'status': 'discovered', 'message': 'ingested ...",2026-08-07 06:33:04.032830+00:00


In [ ]:
import pandas as pd

api_id = "d1568d06-8dec-4974-9a0d-e7e836137bd7"

query = """
SELECT *
FROM backend_api_table
WHERE api_id = %(api_id)s;
"""

df = pd.read_sql(query, engine, params={"api_id": api_id})

if df.empty:
    print("No API found with this api_id.")
else:
    print(df.to_string(index=False))

                              api_id method             end_point                                     description access_to_use                                                                                                                                                                                                                         parameter                                                 error_status                       created_at
d1568d06-8dec-4974-9a0d-e7e836137bd7    GET /api/v1/account/roles Executes GetRoles. Usable by AccountApi agents.       allowed {'priority': 100, 'tool_name': 'get_roles', 'risk_level': 'MEDIUM', 'auto_register': True, 'operation_type': 'GET', 'business_domain': 'Unassigned', 'agent_accessible': True, 'restricted_agents': [], 'recommended_agents': []} {'status': 'discovered', 'message': 'ingested from swagger'} 2026-08-07 06:33:04.032830+00:00


In [7]:
import pandas as pd
import requests
from urllib.parse import urljoin

BASE_URL = "https://localhost:7204"

TOKEN = "eyJhbGciOiJodHRwOi8vd3d3LnczLm9yZy8yMDAxLzA0L3htbGRzaWctbW9yZSNobWFjLXNoYTI1NiIsInR5cCI6IkpXVCJ9.eyJodHRwOi8vc2NoZW1hcy54bWxzb2FwLm9yZy93cy8yMDA1LzA1L2lkZW50aXR5L2NsYWltcy9uYW1lIjoiNTEiLCJodHRwOi8vc2NoZW1hcy54bWxzb2FwLm9yZy93cy8yMDA1LzA1L2lkZW50aXR5L2NsYWltcy9uYW1laWRlbnRpZmllciI6IklJSU9UIFRlYW0iLCJlbWFpbCI6IklJSU9UIFRlYW0iLCJFbXBsb3llZU5hbWUiOiIiLCJjb21wYW55SWQiOiIxMyIsImRlcGFydG1lbnQiOiIiLCJmdW5jdGlvbklkIjoiMCIsImRlcGFydG1lbnRJZCI6IjAiLCJlbXBsb3llZUlkIjoiIiwiVXNlcklkIjoiNTEiLCJodHRwOi8vc2NoZW1hcy5taWNyb3NvZnQuY29tL3dzLzIwMDgvMDYvaWRlbnRpdHkvY2xhaW1zL3JvbGUiOlsiU0FMRVNfVEVBTSIsIlNGR01hbmFnZXIiLCJEaXNwYXRjaFRlYW0iLCJDb21wYW55IiwiU0ZHTWFuYWdlciIsIlFDIiwiUHJvZHVjdGlvblN0b3JlSW5jaGFyZ2UiLCJWZWhpY2xlTWFuYWdlciIsIk1hbmFnZXIiLCJPcGVyYXRvciIsIldJUFN0b3JhZ2VMb2NhdGlvbiIsIk1hY2hpbmUgMiIsIk1hY2hpbmUgMyIsIkJBU0lDX0FDQ0VTUyJdLCJleHAiOjE3ODY1MTMyNjd9.6T2h7sVdWcA0p4iv1cWuSQJ6muGrnx6I59koOGTRFQ4"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json"
}

api_id = "d1568d06-8dec-4974-9a0d-e7e836137bd7"

# Get API details from PostgreSQL
query = """
SELECT method, end_point
FROM backend_api_table
WHERE api_id = %(api_id)s;
"""

api_df = pd.read_sql(query, engine, params={"api_id": api_id})

if api_df.empty:
    raise Exception("API not found")

method = api_df.loc[0, "method"].upper()
endpoint = api_df.loc[0, "end_point"]

print(f"Method   : {method}")
print(f"Endpoint : {endpoint}")

url = urljoin(BASE_URL, endpoint.lstrip("/"))

# Call API
response = requests.request(
    method=method,
    url=url,
    headers=headers,
    verify=False
)

print("Status:", response.status_code)

try:
    print(response.json())
except Exception:
    print(response.text)

Method   : GET
Endpoint : /api/v1/account/roles
Status: 200
[{'roleId': 1, 'roleName': 'SuperAdmin', 'displayName': 'Super Admin', 'createdBy': 'testuser', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 2, 'roleName': 'GateKeeper', 'displayName': 'Gate Keeper', 'createdBy': 'testuser', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 3, 'roleName': 'CustomManager', 'displayName': 'Custom Manager', 'createdBy': 'testuser', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 4, 'roleName': 'WareHouseManager', 'displayName': 'Ware House Manager', 'createdBy': 'test', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 5, 'roleName': 'GateAdmin', 'displayName': 'Gate Admin', 'createdBy': 'test', 'createdDate': '2023-10-10T00:00:00', 'updatedDate'